
# Move segmentation JSON files into the same organized folders as their images

Goal: after running this, every JSON annotation file should sit next to its matching image inside the already-organized folder structure, for example:

```text
organized_output/
├── usable/
│   ├── IMG_001.png
│   └── IMG_001.json
├── unusable/
│   └── polish/
│       ├── IMG_002.png
│       └── IMG_002.json
└── anomaly/
    └── zero_bytes/
        ├── IMG_003.png
        └── IMG_003.json
```

This notebook **does not edit JSON contents**. It only moves/copies JSON files.



## 1. Settings

Change these paths first.

- `JSON_FOLDER`: folder containing all your segmentation `.json` files.
- `ORGANIZED_IMAGE_FOLDER`: the already organized image folder containing `usable/`, `unusable/`, and `anomaly/`.
- `SHEET_FILES`: your CSV/Excel files with image names and reasons. These are optional for matching, but useful for reports/fallback.

Recommended first run:

```python
DRY_RUN = True
```

After the preview looks correct, change to:

```python
DRY_RUN = False
```


In [3]:

from pathlib import Path

# ========= CHANGE THESE =========
JSON_FOLDER = Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/json')
ORGANIZED_IMAGE_FOLDER = Path(r'/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy')

# Optional: CSV / Excel files that contain image names + reasons.
# You can keep these as [] if you only want to match JSONs to images already inside ORGANIZED_IMAGE_FOLDER.
SHEET_FILES = [
    Path(r"/PUT/YOUR/excluded.csv/HERE"),
    Path(r"/PUT/YOUR/inventory_saved.csv/HERE"),
]

# True = preview only. False = actually move/copy JSON files.
DRY_RUN = False

# If True, JSON files are copied instead of moved. Safer, but original JSON folder will still contain them.
# If your goal is "no JSON left outside organized folders", set this to False after preview.
COPY_INSTEAD_OF_MOVE = False

# If a destination JSON already exists:
# - "skip" = do not overwrite
# - "overwrite" = replace destination
# - "rename" = save as filename__duplicate_001.json
ON_CONFLICT = "skip"

# File report output
REPORT_DIR = ORGANIZED_IMAGE_FOLDER / "_reports_json_move"


## 2. Imports and helper functions

In [4]:

import os
import re
import json
import shutil
import hashlib
from datetime import datetime
import pandas as pd

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp", ".heic", ".heif"}

USABLE_REASONS = {"", "nan", "none", "null"}
UNUSABLE_REASONS = {"polish", "b/w", "bw", "black white", "black/white", "occluded", "pathology", "blurry"}
ANOMALY_REASONS = {"anomaly", "unknown", "zero bytes", "zero_bytes", "zero-byte", "zerobytes"}

REASON_FOLDER_MAP = {
    "": ("usable", ""),
    "nan": ("usable", ""),
    "none": ("usable", ""),
    "null": ("usable", ""),
    "polish": ("unusable", "polish"),
    "clear polish": ("unusable", "polish"),
    "polish (1 finger available)": ("unusable", "polish"),
    "b/w": ("unusable", "b_w"),
    "bw": ("unusable", "b_w"),
    "black white": ("unusable", "b_w"),
    "black/white": ("unusable", "b_w"),
    "occluded": ("unusable", "occluded"),
    "pathology": ("unusable", "pathology"),
    "blurry": ("unusable", "blurry"),
    "anomaly": ("anomaly", "anomaly"),
    "unknown": ("anomaly", "anomaly"),
    "zero bytes": ("anomaly", "zero_bytes"),
    "zero_bytes": ("anomaly", "zero_bytes"),
    "zero-byte": ("anomaly", "zero_bytes"),
    "zerobytes": ("anomaly", "zero_bytes"),
}

def norm_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def norm_key(x):
    return norm_text(x).lower()

def normalize_reason(reason):
    r = norm_key(reason)
    # remove extra notes after comma/semicolon if needed, but keep common phrases first
    if r in REASON_FOLDER_MAP:
        return r
    # fuzzy contains rules
    if "zero" in r and "byte" in r:
        return "zero bytes"
    if "polish" in r:
        return "polish"
    if r in {"b w", "black and white", "grayscale", "grey scale", "gray scale"}:
        return "b/w"
    if "blur" in r:
        return "blurry"
    if "occlud" in r or "block" in r:
        return "occluded"
    if "patholog" in r or "disease" in r:
        return "pathology"
    if "anomal" in r or "unknown" in r:
        return "anomaly"
    return r

def classify_reason(reason):
    nr = normalize_reason(reason)
    if nr in REASON_FOLDER_MAP:
        top, sub = REASON_FOLDER_MAP[nr]
        return top, sub, nr
    # unknown nonblank reason is treated as anomaly/anomaly so it does not silently become usable
    if nr:
        return "anomaly", "anomaly", nr
    return "usable", "", nr

def read_table_flexible(path: Path):
    path = Path(path)
    if not path.exists():
        print(f"Missing sheet, skipped: {path}")
        return None
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path)
    encodings = ["utf-8-sig", "utf-8", "cp950", "big5", "latin1"]
    last_err = None
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception as e:
            last_err = e
    raise RuntimeError(f"Could not read {path}. Last error: {last_err}")

def sha256_file(path: Path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

def safe_destination(dest: Path, on_conflict="skip"):
    if not dest.exists():
        return dest, "ok"
    if on_conflict == "overwrite":
        return dest, "overwrite"
    if on_conflict == "skip":
        return dest, "skip_existing"
    if on_conflict == "rename":
        base = dest.with_suffix("")
        suffix = dest.suffix
        for i in range(1, 10000):
            candidate = Path(f"{base}__duplicate_{i:03d}{suffix}")
            if not candidate.exists():
                return candidate, "renamed"
        return dest, "rename_failed"
    raise ValueError("ON_CONFLICT must be 'skip', 'overwrite', or 'rename'")

def extract_imagepath_from_labelme_json(json_path: Path):
    try:
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        image_path = data.get("imagePath", "")
        if image_path:
            return Path(str(image_path)).name, Path(str(image_path)).stem
    except Exception:
        return "", ""
    return "", ""

def candidate_json_keys(json_path: Path):
    keys = []
    stem = json_path.stem
    keys.append(stem.lower())
    image_name, image_stem = extract_imagepath_from_labelme_json(json_path)
    if image_stem:
        keys.append(image_stem.lower())
    if image_name:
        keys.append(Path(image_name).stem.lower())
    # fallback: strip common segmentation/crop suffixes if present
    stripped = re.sub(r"(_nail_?\d+|_crop_?\d+|_seg_?\d+|_mask_?\d+)$", "", stem, flags=re.IGNORECASE)
    if stripped and stripped != stem:
        keys.append(stripped.lower())
    # preserve order, remove duplicates
    seen = set()
    out = []
    for k in keys:
        if k and k not in seen:
            seen.add(k)
            out.append(k)
    return out


## 3. Load optional CSV/Excel sheet mapping

In [5]:

# This sheet mapping is used only as fallback/report context.
# The main placement is based on where the matching image already exists inside ORGANIZED_IMAGE_FOLDER.

sheet_rows = []
for sheet in SHEET_FILES:
    if not str(sheet) or "PUT/YOUR" in str(sheet):
        continue
    df = read_table_flexible(sheet)
    if df is None:
        continue
    df.columns = [str(c).strip() for c in df.columns]
    print(f"Loaded {sheet.name}: {df.shape}")
    print("Columns:", list(df.columns))

    for _, row in df.iterrows():
        # Try common columns from your files
        basename = ""
        stem = ""
        reason = ""
        notes = ""
        for col in ["basename", "file", "filename", "image", "image_name", "name"]:
            if col in df.columns and norm_text(row.get(col)):
                basename = norm_text(row.get(col))
                break
        for col in ["stem", "stem_lower"]:
            if col in df.columns and norm_text(row.get(col)):
                stem = norm_text(row.get(col))
                break
        if not stem and basename:
            stem = Path(basename).stem
        for col in ["reason", "label", "category"]:
            if col in df.columns:
                reason = norm_text(row.get(col))
                break
        for col in ["notes", "note", "comment", "comments"]:
            if col in df.columns:
                notes = norm_text(row.get(col))
                break
        top, sub, nr = classify_reason(reason)
        if stem:
            sheet_rows.append({
                "stem_key": stem.lower(),
                "stem": stem,
                "basename": basename,
                "reason_original": reason,
                "reason_normalized": nr,
                "top_folder": top,
                "reason_folder": sub,
                "notes": notes,
                "source_sheet": sheet.name,
            })

sheet_map = {}
for r in sheet_rows:
    # excluded sheets usually have actual nonblank reasons; let later rows overwrite only if earlier reason blank
    k = r["stem_key"]
    if k not in sheet_map or (not sheet_map[k].get("reason_original") and r.get("reason_original")):
        sheet_map[k] = r

print(f"Sheet rows loaded: {len(sheet_rows)}")
print(f"Unique sheet stems: {len(sheet_map)}")
if sheet_rows:
    display(pd.DataFrame(sheet_rows).head(10))


Sheet rows loaded: 0
Unique sheet stems: 0


## 4. Scan the organized image folder

In [6]:

assert ORGANIZED_IMAGE_FOLDER.exists(), f"ORGANIZED_IMAGE_FOLDER does not exist: {ORGANIZED_IMAGE_FOLDER}"
assert JSON_FOLDER.exists(), f"JSON_FOLDER does not exist: {JSON_FOLDER}"

image_paths = []
for p in ORGANIZED_IMAGE_FOLDER.rglob("*"):
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
        # ignore reports folders if any image accidentally appears there
        if any(part.startswith("_reports") for part in p.parts):
            continue
        image_paths.append(p)

image_index = {}
duplicate_image_keys = {}
for p in image_paths:
    key = p.stem.lower()
    if key in image_index:
        duplicate_image_keys.setdefault(key, [image_index[key]]).append(p)
    else:
        image_index[key] = p

print(f"Organized image folder: {ORGANIZED_IMAGE_FOLDER}")
print(f"Images found: {len(image_paths)}")
print(f"Unique image stems: {len(image_index)}")
print(f"Duplicate image stems: {len(duplicate_image_keys)}")

# Folder count preview
folder_counts = {}
for p in image_paths:
    rel_parent = str(p.parent.relative_to(ORGANIZED_IMAGE_FOLDER))
    folder_counts[rel_parent] = folder_counts.get(rel_parent, 0) + 1
folder_counts_df = pd.DataFrame(sorted(folder_counts.items()), columns=["folder", "image_count"])
display(folder_counts_df)


Organized image folder: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy
Images found: 1007
Unique image stems: 1007
Duplicate image stems: 0


,folder,image_count
0,anomaly/anomaly,7
1,anomaly/zero_bytes,43
2,unusable/blurry,3
3,unusable/occluded,3
4,unusable/pathology,7
5,unusable/polish,108
6,usable,836


## 5. Scan JSON files and decide destination

In [7]:

json_paths = [p for p in JSON_FOLDER.rglob("*.json") if p.is_file()]
print(f"JSON folder: {JSON_FOLDER}")
print(f"JSON files found: {len(json_paths)}")

plan_rows = []
for jp in json_paths:
    keys = candidate_json_keys(jp)
    matched_image = None
    match_key = ""
    match_method = ""
    for k in keys:
        if k in image_index:
            matched_image = image_index[k]
            match_key = k
            match_method = "organized_image_stem_or_imagePath"
            break

    sheet_info = None
    for k in keys:
        if k in sheet_map:
            sheet_info = sheet_map[k]
            break

    if matched_image is not None:
        dest_dir = matched_image.parent
        dest_path = dest_dir / jp.name
        target_status = "matched_to_existing_organized_image"
        top_folder = dest_dir.relative_to(ORGANIZED_IMAGE_FOLDER).parts[0] if dest_dir != ORGANIZED_IMAGE_FOLDER else ""
        reason_folder = str(dest_dir.relative_to(ORGANIZED_IMAGE_FOLDER))
    elif sheet_info is not None:
        # fallback target from sheet only; useful if images are not present yet
        top = sheet_info["top_folder"]
        sub = sheet_info["reason_folder"]
        dest_dir = ORGANIZED_IMAGE_FOLDER / top / sub if sub else ORGANIZED_IMAGE_FOLDER / top
        dest_path = dest_dir / jp.name
        target_status = "fallback_from_sheet_no_matching_image_found"
        top_folder = top
        reason_folder = sub
    else:
        dest_dir = None
        dest_path = None
        target_status = "unmatched_no_image_no_sheet_row"
        top_folder = ""
        reason_folder = ""

    already_with_image = False
    if matched_image is not None:
        already_with_image = (jp.parent.resolve() == matched_image.parent.resolve())

    plan_rows.append({
        "json_path": str(jp),
        "json_name": jp.name,
        "json_stem": jp.stem,
        "candidate_keys": " | ".join(keys),
        "target_status": target_status,
        "match_key": match_key,
        "match_method": match_method,
        "matched_image_path": str(matched_image) if matched_image else "",
        "dest_path": str(dest_path) if dest_path else "",
        "already_with_image": already_with_image,
        "top_folder": top_folder,
        "reason_folder": reason_folder,
        "sheet_reason_original": sheet_info.get("reason_original", "") if sheet_info else "",
        "sheet_reason_normalized": sheet_info.get("reason_normalized", "") if sheet_info else "",
        "source_sheet": sheet_info.get("source_sheet", "") if sheet_info else "",
    })

plan_df = pd.DataFrame(plan_rows)
print("Decision summary:")
display(plan_df["target_status"].value_counts(dropna=False).rename_axis("target_status").reset_index(name="count"))
print("Already with matching image:", int(plan_df["already_with_image"].sum()) if len(plan_df) else 0)
display(plan_df.head(20))


JSON folder: /Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/json
JSON files found: 949
Decision summary:


,target_status,count
0,matched_to_existing_organized_image,949


Already with matching image: 0


,json_path,json_name,json_stem,candidate_keys,target_status,match_key,match_method,matched_image_path,dest_path,already_with_image,top_folder,reason_folder,sheet_reason_original,sheet_reason_normalized,source_sheet
0,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20220328152405_pid2625_4apaJV.json,20220328152405_pid2625_4apaJV,20220328152405_pid2625_4apajv,matched_to_existing_organized_image,20220328152405_pid2625_4apajv,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
1,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221103111926_pid2625_2PqKf8.json,20221103111926_pid2625_2PqKf8,20221103111926_pid2625_2pqkf8,matched_to_existing_organized_image,20221103111926_pid2625_2pqkf8,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
2,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221215105320_pid2625_JdWIRq.json,20221215105320_pid2625_JdWIRq,20221215105320_pid2625_jdwirq,matched_to_existing_organized_image,20221215105320_pid2625_jdwirq,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
3,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221111110046_pid2625_MpxKZK.json,20221111110046_pid2625_MpxKZK,20221111110046_pid2625_mpxkzk,matched_to_existing_organized_image,20221111110046_pid2625_mpxkzk,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,unusable,unusable/polish,,,
4,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221018151019_pid2625_rxbmqf.json,20221018151019_pid2625_rxbmqf,20221018151019_pid2625_rxbmqf,matched_to_existing_organized_image,20221018151019_pid2625_rxbmqf,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,anomaly,anomaly/anomaly,,,
5,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20211203094314_pid2625_evGttu.json,20211203094314_pid2625_evGttu,20211203094314_pid2625_evgttu,matched_to_existing_organized_image,20211203094314_pid2625_evgttu,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
6,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20211110145713_pid2625_omm827.json,20211110145713_pid2625_omm827,20211110145713_pid2625_omm827,matched_to_existing_organized_image,20211110145713_pid2625_omm827,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
7,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20230228174103_pid2625_8ZtNaV.json,20230228174103_pid2625_8ZtNaV,20230228174103_pid2625_8ztnav,matched_to_existing_organized_image,20230228174103_pid2625_8ztnav,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
8,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20220822110025_pid2625_WMY4Li.json,20220822110025_pid2625_WMY4Li,20220822110025_pid2625_wmy4li,matched_to_existing_organized_image,20220822110025_pid2625_wmy4li,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,
9,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221104142131_pid2625_TJMxoX.json,20221104142131_pid2625_TJMxoX,20221104142131_pid2625_tjmxox,matched_to_existing_organized_image,20221104142131_pid2625_tjmxox,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,


## 6. Preview problems before moving

In [8]:

# JSONs that could not be placed safely
unmatched_df = plan_df[plan_df["target_status"].eq("unmatched_no_image_no_sheet_row")].copy()
print(f"Unmatched JSON files: {len(unmatched_df)}")
if len(unmatched_df):
    display(unmatched_df[["json_name", "json_path", "candidate_keys"]].head(50))

# JSONs where destination already exists
conflict_rows = []
for _, r in plan_df.iterrows():
    dest = r["dest_path"]
    if dest and Path(dest).exists() and Path(r["json_path"]).resolve() != Path(dest).resolve():
        conflict_rows.append(r)
conflict_df = pd.DataFrame(conflict_rows)
print(f"Destination conflicts: {len(conflict_df)}")
if len(conflict_df):
    display(conflict_df[["json_name", "json_path", "dest_path", "matched_image_path"]].head(50))

print("DRY_RUN =", DRY_RUN)
print("COPY_INSTEAD_OF_MOVE =", COPY_INSTEAD_OF_MOVE)
print("ON_CONFLICT =", ON_CONFLICT)


Unmatched JSON files: 0
Destination conflicts: 0
DRY_RUN = False
COPY_INSTEAD_OF_MOVE = False
ON_CONFLICT = skip


## 7. Move/copy JSON files

In [9]:

REPORT_DIR.mkdir(parents=True, exist_ok=True)

action_rows = []
for _, r in plan_df.iterrows():
    src = Path(r["json_path"])
    dest_str = r["dest_path"]
    if not dest_str:
        action_rows.append({**r.to_dict(), "action_status": "unmatched_not_moved", "final_dest_path": ""})
        continue
    dest = Path(dest_str)

    if not src.exists():
        action_rows.append({**r.to_dict(), "action_status": "source_missing", "final_dest_path": str(dest)})
        continue

    # Already exactly in target place
    if src.resolve() == dest.resolve():
        action_rows.append({**r.to_dict(), "action_status": "already_in_place", "final_dest_path": str(dest)})
        continue

    final_dest, conflict_action = safe_destination(dest, ON_CONFLICT)
    if conflict_action == "skip_existing":
        action_rows.append({**r.to_dict(), "action_status": "skipped_existing_destination", "final_dest_path": str(final_dest)})
        continue
    if conflict_action == "rename_failed":
        action_rows.append({**r.to_dict(), "action_status": "rename_failed", "final_dest_path": str(final_dest)})
        continue

    if DRY_RUN:
        action_rows.append({**r.to_dict(), "action_status": f"dry_run_would_{'copy' if COPY_INSTEAD_OF_MOVE else 'move'}", "final_dest_path": str(final_dest)})
        continue

    final_dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        if conflict_action == "overwrite" and final_dest.exists():
            final_dest.unlink()
        if COPY_INSTEAD_OF_MOVE:
            shutil.copy2(src, final_dest)
            action_status = "copied"
        else:
            shutil.move(str(src), str(final_dest))
            action_status = "moved"
        action_rows.append({**r.to_dict(), "action_status": action_status, "final_dest_path": str(final_dest)})
    except Exception as e:
        action_rows.append({**r.to_dict(), "action_status": f"error: {type(e).__name__}: {e}", "final_dest_path": str(final_dest)})

action_df = pd.DataFrame(action_rows)
print("Action summary:")
display(action_df["action_status"].value_counts(dropna=False).rename_axis("action_status").reset_index(name="count"))
display(action_df.head(20))


Action summary:


,action_status,count
0,moved,949


,json_path,json_name,json_stem,candidate_keys,target_status,match_key,match_method,matched_image_path,dest_path,already_with_image,top_folder,reason_folder,sheet_reason_original,sheet_reason_normalized,source_sheet,action_status,final_dest_path
0,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20220328152405_pid2625_4apaJV.json,20220328152405_pid2625_4apaJV,20220328152405_pid2625_4apajv,matched_to_existing_organized_image,20220328152405_pid2625_4apajv,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
1,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221103111926_pid2625_2PqKf8.json,20221103111926_pid2625_2PqKf8,20221103111926_pid2625_2pqkf8,matched_to_existing_organized_image,20221103111926_pid2625_2pqkf8,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
2,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221215105320_pid2625_JdWIRq.json,20221215105320_pid2625_JdWIRq,20221215105320_pid2625_jdwirq,matched_to_existing_organized_image,20221215105320_pid2625_jdwirq,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
3,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221111110046_pid2625_MpxKZK.json,20221111110046_pid2625_MpxKZK,20221111110046_pid2625_mpxkzk,matched_to_existing_organized_image,20221111110046_pid2625_mpxkzk,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,unusable,unusable/polish,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
4,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20221018151019_pid2625_rxbmqf.json,20221018151019_pid2625_rxbmqf,20221018151019_pid2625_rxbmqf,matched_to_existing_organized_image,20221018151019_pid2625_rxbmqf,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,anomaly,anomaly/anomaly,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
5,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20211203094314_pid2625_evGttu.json,20211203094314_pid2625_evGttu,20211203094314_pid2625_evgttu,matched_to_existing_organized_image,20211203094314_pid2625_evgttu,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
6,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20211110145713_pid2625_omm827.json,20211110145713_pid2625_omm827,20211110145713_pid2625_omm827,matched_to_existing_organized_image,20211110145713_pid2625_omm827,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
7,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20230228174103_pid2625_8ZtNaV.json,20230228174103_pid2625_8ZtNaV,20230228174103_pid2625_8ztnav,matched_to_existing_organized_image,20230228174103_pid2625_8ztnav,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NTHU 3.2/special to...,False,usable,usable,,,,moved,/Users/williamtsai/Desktop/NTHU 3.2/special to...
8,/Users/williamtsai/Desktop/NTHU 3.2/special to...,20220822110025_pid2625_WMY4Li.json,20220822110025_pid2625_WMY4Li,20220822110025_pid2625_wmy4li,matched_to_existing_organized_image,20220822110025_pid2625_wmy4li,organized_image_stem_or_imagePath,/Users/williamtsai/Desktop/NTHU 3.2/special to...,/Users/williamtsai/Desktop/NT

## 8. Verify: every organized image has JSON if available, and every moved JSON is with image

In [10]:

# Re-scan after moving/copying
organized_json_paths = [p for p in ORGANIZED_IMAGE_FOLDER.rglob("*.json") if p.is_file()]
organized_json_index = {}
for p in organized_json_paths:
    organized_json_index.setdefault(p.stem.lower(), []).append(p)

verify_rows = []
for img in image_paths:
    expected_json = img.with_suffix(".json")
    same_folder_jsons = [p for p in img.parent.glob("*.json") if p.stem.lower() == img.stem.lower()]
    verify_rows.append({
        "image_path": str(img),
        "image_name": img.name,
        "image_stem": img.stem,
        "image_folder": str(img.parent.relative_to(ORGANIZED_IMAGE_FOLDER)),
        "expected_json_path": str(expected_json),
        "has_matching_json_in_same_folder": len(same_folder_jsons) > 0,
        "matching_json_paths": " | ".join(str(p) for p in same_folder_jsons),
    })

verify_df = pd.DataFrame(verify_rows)
print("Images with matching JSON in same folder:")
if len(verify_df):
    print(f"{verify_df['has_matching_json_in_same_folder'].sum()} / {len(verify_df)}")
    display(verify_df['has_matching_json_in_same_folder'].value_counts(dropna=False).rename_axis('has_json').reset_index(name='count'))
    missing_json_for_images = verify_df[~verify_df["has_matching_json_in_same_folder"]].copy()
    print(f"Images still missing same-folder JSON: {len(missing_json_for_images)}")
    if len(missing_json_for_images):
        display(missing_json_for_images[["image_name", "image_folder", "image_path"]].head(50))

# Check JSONs still left in source JSON_FOLDER after move mode
remaining_source_jsons = [p for p in JSON_FOLDER.rglob("*.json") if p.is_file()]
print(f"JSON files remaining in original JSON_FOLDER: {len(remaining_source_jsons)}")
if (not COPY_INSTEAD_OF_MOVE) and (not DRY_RUN) and len(remaining_source_jsons):
    display(pd.DataFrame({"remaining_json_path": [str(p) for p in remaining_source_jsons]}).head(50))


Images with matching JSON in same folder:
949 / 1007


,has_json,count
0,True,949
1,False,58


Images still missing same-folder JSON: 58


,image_name,image_folder,image_path
0,20221115103130_pid2625_zTEoFR.jpeg,anomaly/anomaly,/Users/williamtsai/Desktop/NTHU 3.2/special to...
2,20221116100613_pid2625_7Ckw9S.jpeg,anomaly/anomaly,/Users/williamtsai/Desktop/NTHU 3.2/special to...
6,20221115122636_pid2625_zVKthD.jpeg,anomaly/anomaly,/Users/williamtsai/Desktop/NTHU 3.2/special to...
7,20210701131914_pid2625_9NJTei.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...
8,20210630164248_pid2625_GtYap9.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...
9,20210713163949_pid2625_bCraVA.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...
10,20210629145732_pid2625_985urE.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...
11,20210709165700_pid2625_uCjVbj.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...
12,20210701135425_pid2625_gIogZd.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...
13,20210701124802_pid2625_9niBUk.jpg,anomaly/zero_bytes,/Users/williamtsai/Desktop/NTHU 3.2/special to...


JSON files remaining in original JSON_FOLDER: 0


## 9. Save reports

In [11]:

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
plan_report = REPORT_DIR / f"json_move_plan_report_{ts}.csv"
action_report = REPORT_DIR / f"json_move_action_report_{ts}.csv"
verify_report = REPORT_DIR / f"json_move_verify_report_{ts}.csv"

plan_df.to_csv(plan_report, index=False, encoding="utf-8-sig")
action_df.to_csv(action_report, index=False, encoding="utf-8-sig")
verify_df.to_csv(verify_report, index=False, encoding="utf-8-sig")

print("Saved reports:")
print(plan_report)
print(action_report)
print(verify_report)


Saved reports:
/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/_reports_json_move/json_move_plan_report_20260619_225751.csv
/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/_reports_json_move/json_move_action_report_20260619_225751.csv
/Users/williamtsai/Desktop/NTHU 3.2/special topic/will data/organizing/eyeimage5_organized_exact_copy/_reports_json_move/json_move_verify_report_20260619_225751.csv



## How to know it worked

Open the action report:

```text
_reports_json_move/json_move_action_report_*.csv
```

Good statuses:

- `moved`
- `copied`
- `already_in_place`

Problem statuses to inspect:

- `unmatched_not_moved`
- `skipped_existing_destination`
- `source_missing`
- `error: ...`

Then open the verification report:

```text
_reports_json_move/json_move_verify_report_*.csv
```

The important column is:

```text
has_matching_json_in_same_folder
```

For images that have JSON annotations, this should be `True`.
